In [ ]:
import pickle
import os
binary = os.path.join(".", "outputs", "baseline-test-function_name_idx_map.pkl")
with open(binary, 'rb') as c:
    data = pickle.load(c)

In [7]:
import re
bin_name = "output_x86-gcc-7-O3_minigzip64"
re.split(r"[-_]", bin_name)

['output', 'x86', 'gcc', '7', 'O3', 'minigzip64']

In [3]:
# function_name, compiler, version, optimization, binary_name = 'get_peeraddr_string', 'gcc', '9', 'O3', 'ncat'
data[('get_peeraddr_string','gcc','9','O3','ncat')]

KeyError: ('get_peeraddr_string', 'gcc', '9', 'O3', 'ncat')

In [7]:
import pickle
import os
data_dir = "."
dataset_name = "data_split"
binary = os.path.join(data_dir, "outputs", f"{dataset_name}.pkl")
with open(binary, 'rb') as c:
    data = pickle.load(c)
print(data.keys())
list(set(data['test']).union(data['valid']).union(data['train']))

dict_keys(['train', 'valid', 'test'])


['ossltest.so',
 'afalg.so',
 'clamscan',
 'nping',
 'sigtool',
 'openssl',
 'curl',
 'clambc',
 'minigzipsh',
 'dasync.so',
 'capi.so',
 'padlock.so',
 'libcrypto.so.3',
 'libssl.so.3',
 'nmap',
 'legacy.so',
 'z3',
 'libz.so.1.2.11',
 'clamconf',
 'ncat',
 'minigzip64',
 'minigzip',
 'unrar',
 'freshclam',
 'fips.so',
 'libclamav.so.9.0.0']

In [ ]:
import pandas
import os
data_path = os.path.join(".", "outputs", "function_pools.csv")
data = pandas.read_csv(data_path, dtype={"anchor_version": str, "target_version": str},)

In [12]:
set(data['anchor_function_bin'].unique().tolist() + data['target_function_bin'].unique().tolist())

{'afalg.so',
 'clamscan',
 'curl',
 'dasync.so',
 'freshclam',
 'legacy.so',
 'libcrypto.so.3',
 'libssl.so.3',
 'ncat',
 'nmap',
 'openssl',
 'ossltest.so',
 'padlock.so',
 'sigtool',
 'unrar',
 'z3'}

In [19]:
data[['anchor_function_bin', 'anchor_compiler', 'anchor_version', 'anchor_opt']].groupby(['anchor_function_bin', 'anchor_compiler', 'anchor_version', 'anchor_opt']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,anchor_function_bin,anchor_compiler,anchor_version,anchor_opt,count
366,z3,gcc,5,O0,97200
377,z3,gcc,9,O3,97000
354,z3,clang,7,O0,94200
372,z3,gcc,7,O2,93600
373,z3,gcc,7,O3,93200
...,...,...,...,...,...
4,afalg.so,clang,9,O1,200
3,afalg.so,clang,7,O1,200
2,afalg.so,clang,5,O2,200
1,afalg.so,clang,3.5,O3,200


In [2]:
from datasets import load_dataset
from models.collatefn import MLM_ANP_CollateFn
import os
import torch
from torch.utils.data import DataLoader
from models.tokenizer import AsmTokenizer
import numpy as np
import json
data_dir = "."
dataset_name = "baseline-train"
# dataset_name = "baseline-valid"
# dataset_name = "baseline-test"
dataset_path = os.path.join(data_dir, "outputs", f"{dataset_name}.jsonl")
metadata_path = os.path.join(data_dir, "outputs", f"{dataset_name}-metadata.jsonl")
with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)
    dataset_size = metadata['__metadata__']['dataset_size']
    print(f"Dataset size: {dataset_size}")

dataset = load_dataset('json', data_files=dataset_path, split='train', streaming=True)
dataset._info.dataset_size = dataset_size
shuffled_dataset = dataset.shuffle(seed=42, buffer_size=5000)
tokenizer = AsmTokenizer(vocab_file=os.path.join(data_dir, "outputs", f"baseline-vocab.txt"))

dataloader = DataLoader(
    shuffled_dataset,
    batch_size=256,           # 批次大小
    collate_fn=MLM_ANP_CollateFn(tokenizer)     # 使用自定义批处理函数
)
data = next(iter(dataloader))


Dataset size: 1030443
Vocab loaded from .\outputs\baseline-vocab.txt


In [3]:
mini_batch_size = 256
mlm_input_ids = data["mlm_input_ids"]
# mlm_labels = data["mlm_labels"]
# anp_input_ids_a = data["anp_input_ids_a"]
# anp_input_ids_b = data["anp_input_ids_b"]
# anp_input_labels = data["anp_labels"]
iter_mlm_input_ids = torch.split(mlm_input_ids, mini_batch_size)

In [7]:
iter_mlm_input_ids[0]

tensor([[ 1, 21, 12,  ...,  0,  0,  0],
        [ 1, 16, 55,  ...,  0,  0,  0],
        [ 1, 16, 11,  ...,  0,  0,  0],
        ...,
        [ 1, 16,  9,  ...,  0,  0,  0],
        [ 1, 16,  9,  ...,  0,  0,  0],
        [ 1, 16,  9,  ...,  0,  0,  0]])

In [ ]:
def random_word_parallel(tokens):
    # convert tokens to numpy array for random operations
    tokens_arr = np.array(tokens)
    
    # Randomly mask some tokens
    mask_prob = np.random.rand(len(tokens_arr)) < 0.15

    # generate a random strategy for each token
    # < 8: replace with <MASK>
    # == 8: replace with random token
    # > 8: keep original token
    strategy = np.random.randint(0, 10, size=len(tokens_arr))

    output = np.copy(tokens_arr)
    labels = np.zeros_like(tokens_arr)

    # get the mask token id
    mask_token_id = tokenizer.vocab['<MASK>']
    # get the random token ids
    random_tokens = np.random.randint(0, len(tokenizer.vocab), size=len(tokens_arr))
    # apply the masking strategy
    if np.any(mask_prob):
        # create boolean masks for each strategy
        mask_strategy = mask_prob & (strategy < 8)    # 80% MASK
        rand_strategy = mask_prob & (strategy == 8)   # 10% 随机词
        
        # apply the strategies
        output[mask_strategy] = mask_token_id
        output[rand_strategy] = random_tokens[rand_strategy]
        
        # set the labels
        labels[mask_prob] = tokens_arr[mask_prob]
    assert(len(output) == len(labels))
    return output.tolist(), labels.tolist()

In [ ]:
import random
def random_word_loop(tokens):
    output = []
    labels = []
    for token in tokens:
        if random.random() < 0.15:
            random_choice = random.random()
            if random_choice < 0.8:
                output.append(tokenizer.vocab['<MASK>'])  # 80% Replace with MASK
            elif random_choice < 0.9:
                output.append(random.choice(list(tokenizer.vocab.values())))  # 10% Random token
            else:
                output.append(token)  # 10% Keep original
            labels.append(token)
        else:
            output.append(token)
            labels.append(0)
    assert(len(output) == len(labels))
    return output, labels

In [ ]:
import time

tokens = [5, 55, 10, 33, 8, 5]
start = time.time()
output_parallel, labels_parallel = random_word_parallel(tokens)
end = time.time()
print(f"random_word_parallel time: {end - start:.6f}s")

start = time.time()
output_loop, labels_loop = random_word_loop(tokens)
end = time.time()
print(f"random_word_loop time: {end - start:.6f}s")